# Cluster Overview

This notebook queries the OCP audit SQLite datastore to present the **Cluster Overview** report — a single-row identity card per cluster covering:

- Version & Identity
- Platform & Topology
- Node Sizing
- Network Configuration
- Access Endpoints
- Update Posture

In [1]:
import os
import sys

import pandas as pd

# Shared notebook helpers (sys.path + OCP_AUDIT_DB + styling)
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from notebook_style import bootstrap, style_table  # noqa: E402

session, engine = bootstrap()

from schema.models import Cluster, ClusterOverview  # noqa: E402

print(f"Connected to: {engine.url}")


Connected to: sqlite:////home/vagrant/git/openshift-csv-exporter/datastore/ocp_audit.db


## Cluster Inventory

In [2]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

3 cluster(s) in dataset


env,friendly_name,id,cluster_name,cluster_context,cluster_server
prod,Prod East,1,ocp-prod-east,admin/api-ocp-prod-east:6443,https://api.ocp-prod-east.example.com:6443
stage,Staging,2,ocp-staging,admin/api-ocp-staging:6443,https://api.ocp-staging.example.com:6443
dev,Dev,3,ocp-dev,admin/api-ocp-dev:6443,https://api.ocp-dev.example.com:6443


---
## Version & Identity

OCP version, Kubernetes version, cluster ID, install date, and cluster age.

In [3]:
df_version = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_version)

env,friendly_name,cluster_name,ocp_version,kubernetes_version,cluster_id_ocp,install_date,cluster_age_days
prod,Prod East,ocp-prod-east,4.18.28,v1.31.5,8a7b6c5d-1111-2222-3333-444455556666,2024-01-15T08:30:00Z,443
stage,Staging,ocp-staging,4.18.24,v1.31.4,9b8c7d6e-aaaa-bbbb-cccc-ddddeeeeffff,2024-03-02T12:10:00Z,397
dev,Dev,ocp-dev,4.18.18,v1.31.3,1a2b3c4d-5e6f-7a8b-9c0d-1e2f3a4b5c6d,2024-06-20T09:45:00Z,287


---
## Platform & Topology

Infrastructure platform and control-plane / infrastructure topology.

In [4]:
df_platform = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_platform)

env,friendly_name,cluster_name,platform,control_plane_topology,infrastructure_topology
prod,Prod East,ocp-prod-east,AWS,HighlyAvailable,HighlyAvailable
stage,Staging,ocp-staging,AWS,HighlyAvailable,HighlyAvailable
dev,Dev,ocp-dev,AWS,SingleReplica,SingleReplica


---
## Node Sizing

Master, worker, infra, and total node counts per cluster.

In [5]:
df_nodes = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_nodes)

env,friendly_name,cluster_name,master_count,worker_count,infra_count,total_node_count
prod,Prod East,ocp-prod-east,3,6,3,12
stage,Staging,ocp-staging,3,4,2,9
dev,Dev,ocp-dev,1,2,0,3


---
## Network Configuration

SDN type, cluster CIDRs, and service CIDRs.

In [6]:
df_network = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_network)

env,friendly_name,cluster_name,network_type,cluster_cidrs,service_cidrs
prod,Prod East,ocp-prod-east,OVNKubernetes,10.128.0.0/14,172.30.0.0/16
stage,Staging,ocp-staging,OVNKubernetes,10.128.0.0/14,172.30.0.0/16
dev,Dev,ocp-dev,OVNKubernetes,10.128.0.0/14,172.30.0.0/16


---
## Access Endpoints

Console URL, API server URL, and default ingress domain.

In [7]:
df_endpoints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.default_ingress_domain,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_endpoints)

env,friendly_name,cluster_name,console_url,api_server_url,default_ingress_domain
prod,Prod East,ocp-prod-east,https://console-openshift-console.apps.ocp-prod-east.example.com,https://api.ocp-prod-east.example.com:6443,apps.ocp-prod-east.example.com
stage,Staging,ocp-staging,https://console-openshift-console.apps.ocp-staging.example.com,https://api.ocp-staging.example.com:6443,apps.ocp-staging.example.com
dev,Dev,ocp-dev,https://console-openshift-console.apps.ocp-dev.example.com,https://api.ocp-dev.example.com:6443,apps.ocp-dev.example.com


---
## Update Posture

Current OCP version, update channel, update state, and count of available updates.

In [8]:
df_updates = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.update_channel,
        ClusterOverview.update_state,
        ClusterOverview.available_updates_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_updates)

env,friendly_name,cluster_name,ocp_version,update_channel,update_state,available_updates_count
prod,Prod East,ocp-prod-east,4.18.28,stable-4.18,Completed,0
stage,Staging,ocp-staging,4.18.24,stable-4.18,Completed,2
dev,Dev,ocp-dev,4.18.18,fast-4.18,Completed,5


---
## Full Cluster Overview

All 21 fields for every cluster in a single table.

In [9]:
df_full = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
        ClusterOverview.default_ingress_domain,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.update_channel,
        ClusterOverview.available_updates_count,
        ClusterOverview.update_state,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_full)} cluster overview(s)")
style_table(df_full)

3 cluster overview(s)


env,friendly_name,cluster_name,ocp_version,kubernetes_version,cluster_id_ocp,install_date,cluster_age_days,platform,control_plane_topology,infrastructure_topology,master_count,worker_count,infra_count,total_node_count,network_type,cluster_cidrs,service_cidrs,default_ingress_domain,console_url,api_server_url,update_channel,available_updates_count,update_state
prod,Prod East,ocp-prod-east,4.18.28,v1.31.5,8a7b6c5d-1111-2222-3333-444455556666,2024-01-15T08:30:00Z,443,AWS,HighlyAvailable,HighlyAvailable,3,6,3,12,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-prod-east.example.com,https://console-openshift-console.apps.ocp-prod-east.example.com,https://api.ocp-prod-east.example.com:6443,stable-4.18,0,Completed
stage,Staging,ocp-staging,4.18.24,v1.31.4,9b8c7d6e-aaaa-bbbb-cccc-ddddeeeeffff,2024-03-02T12:10:00Z,397,AWS,HighlyAvailable,HighlyAvailable,3,4,2,9,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-staging.example.com,https://console-openshift-console.apps.ocp-staging.example.com,https://api.ocp-staging.example.com:6443,stable-4.18,2,Completed
dev,Dev,ocp-dev,4.18.18,v1.31.3,1a2b3c4d-5e6f-7a8b-9c0d-1e2f3a4b5c6d,2024-06-20T09:45:00Z,287,AWS,SingleReplica,SingleReplica,1,2,0,3,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-dev.example.com,https://console-openshift-console.apps.ocp-dev.example.com,https://api.ocp-dev.example.com:6443,fast-4.18,5,Completed


In [10]:
session.close()
print("Session closed.")

Session closed.
